# One-vs-Rest (OvR) Strategy for Multi-Class Classification

This notebook demonstrates One-vs-Rest, a decomposition strategy for multi-class problems that trains K binary classifiers for K classes, each solving a binary problem (class vs. rest).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

np.random.seed(42)

## 1. Load and Prepare Data

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
target_names = iris.target_names
n_classes = len(target_names)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Number of classes: {n_classes}")
print(f"Number of OvR binary classifiers: {n_classes}")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2. One-vs-Rest with SVM

In [ ]:
# OvR with SVM (natural multi-class via OvR)
ovr_svm = OneVsRestClassifier(SVC(kernel='rbf', probability=True, random_state=42))
ovr_svm.fit(X_train, y_train)
y_pred_ovr = ovr_svm.predict(X_test)
acc_ovr = accuracy_score(y_test, y_pred_ovr)

print(f"One-vs-Rest SVM Accuracy: {acc_ovr:.4f}")
print(f"Number of estimators (binary classifiers): {len(ovr_svm.estimators_)}")

## 3. Compare with Native Multi-Class

In [ ]:
# Native multi-class SVM
native_svm = SVC(kernel='rbf', decision_function_shape='ovr', random_state=42)
native_svm.fit(X_train, y_train)
y_pred_native = native_svm.predict(X_test)
acc_native = accuracy_score(y_test, y_pred_native)

print("\n=== OvR vs. NATIVE MULTI-CLASS ===")
print(f"One-vs-Rest SVM:     {acc_ovr:.4f}")
print(f"Native Multi-class SVM: {acc_native:.4f}")

## 4. Analyze Binary Classifiers

In [ ]:
# Check individual binary classifier scores
print("\nBinary Classifier Performance (OvR):")
for i, (estimator, target_name) in enumerate(zip(ovr_svm.estimators_, target_names)):
    # Create binary target: class i vs. rest
    y_binary = (y_test == i).astype(int)
    y_pred_binary = estimator.predict(X_test)
    binary_acc = accuracy_score(y_binary, y_pred_binary)
    print(f"  Classifier {i} ({target_name} vs. Rest): {binary_acc:.4f}")

## 5. Compare with Other Strategies

In [ ]:
# Other multi-class approaches
strategies = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, multi_class='multinomial'),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'OvR SVM': OneVsRestClassifier(SVC(kernel='rbf', random_state=42))
}

results = {}
print("\n=== MULTI-CLASS STRATEGY COMPARISON ===")
for name, model in strategies.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"{name:25s}: {acc:.4f}")

## 6. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_ovr)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names, yticklabels=target_names)
plt.title('One-vs-Rest SVM Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 7. Classification Report

In [ ]:
print("\nOne-vs-Rest SVM Classification Report:")
print(classification_report(y_test, y_pred_ovr, target_names=target_names))

## 8. Key Properties of One-vs-Rest

- **Number of classifiers:** K binary classifiers for K classes
- **Imbalanced subproblems:** Each binary problem is imbalanced (class vs. rest)
- **Computational cost:** O(K) training, O(K) prediction
- **Good for:** Linear models (SVM, Logistic Regression), when K is small
- **Risk:** Imbalanced binary problems may hurt performance
- **vs. OvO:** OvR uses K(K-1)/2 classifiers but each is more balanced